In [0]:
access_key = dbutils.secrets.get("healthbot_cred", "healthdata_key")
secret_key = dbutils.secrets.get("healthbot_cred", "healthdata_secret")
print(access_key)
print(secret_key)

spark.conf.set("fs.s3a.access.key", access_key)
spark.conf.set("fs.s3a.secret.key", secret_key)

In [0]:
# sample code to test connection and pull data from aws s3 bucket
# df = spark.read.format("csv").option("header", "true").option("inferschema", "true").load("s3a://health-data-demo-s3/patients_data/")
# df.printSchema()

In [0]:
from pyspark.sql.functions import input_file_name
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os

# ---------------- CONFIG ----------------
S3_FOLDER = "s3a://health-data-demo-s3/"          # where CSVs arrive
ARCHIVE_FOLDER = "s3://YOUR_BUCKET/archive/"     # optional, recommended
CATALOG = "chatbot_dev"
BRONZE_SCHEMA = "bronze"
LOG_TABLE_NAME = "s3_file_ingest_log"

def q(*parts):
    return ".".join([f"`{p}`" for p in parts])

# A small Delta table to track processed files (prevents duplicates)
FILE_LOG_TABLE = q(CATALOG, BRONZE_SCHEMA, LOG_TABLE_NAME)

def sanitize_table_name(filename: str) -> str:
    # "patients_data.csv" -> "patients_data"
    name = os.path.splitext(filename)
    return name

# 1) Ensure schema + log table exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(CATALOG, BRONZE_SCHEMA)}")

# Create file log table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FILE_LOG_TABLE} (
  source_file STRING,
  file_size BIGINT,
  file_mod_time BIGINT,
  ingested_at TIMESTAMP,
  target_table STRING,
  status STRING
)
USING DELTA
""")

print("Created/verified:", FILE_LOG_TABLE)

# 2) Load processed signatures (SUCCESS only)
processed = (
    spark.table(FILE_LOG_TABLE)
         .filter("status = 'SUCCESS'")
         .select("source_file", "file_size", "file_mod_time")
)

processed_keys = set(
    (r["source_file"], r["file_size"], r["file_mod_time"])
    for r in processed.collect()
)

print("Already processed signatures:", len(processed_keys))

# 3) List inbound CSV files
csv_files = [f for f in dbutils.fs.ls(S3_FOLDER) if f.path.lower().endswith(".csv")]

print(f"Found {len(csv_files)} CSV files in {S3_FOLDER}")

# Helper: log ingestion result (explicit schema avoids CANNOT_DETERMINE_TYPE)
log_schema = "source_file STRING, file_size BIGINT, file_mod_time BIGINT, target_table STRING, status STRING"
def log_ingest(source_file, file_size, file_mod_time, target_table, status):
    log_df = (spark.createDataFrame(
                [(source_file, int(file_size), int(file_mod_time), target_table, status)],
                schema=log_schema
             )
             .withColumn("ingested_at", F.current_timestamp())
             .select("source_file","file_size","file_mod_time","ingested_at","target_table","status")
    )
    (log_df.write
          .format("delta")
          .mode("append")
          .saveAsTable(FILE_LOG_TABLE)
    )

# 4) Process each file (idempotent)
for f in csv_files:
    source_file = f.path
    file_size = f.size
    file_mod_time = f.modificationTime
    signature = (source_file, file_size, file_mod_time)

    print("CHECK signature:", signature)

    if signature in processed_keys:
        print(f"SKIP (already ingested): {source_file}")
        continue

    filename = source_file.split("/")[-1]
    table_name, extension = os.path.splitext(filename)
    TARGET_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    print(f"INGEST: {source_file} -> {TARGET_TABLE}")

    try:
        df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .option("mode", "PERMISSIVE")
              .load(source_file)
              .withColumn("ingested_at", F.current_timestamp())
              .withColumn("source_file", F.lit(source_file))
             )

        if df.rdd.isEmpty():
            print(f"SKIP empty content: {source_file}")
            log_ingest(source_file, file_size, file_mod_time, TARGET_TABLE, "SKIPPED_EMPTY")
            continue

        # Append to Bronze (creates table if it doesn't exist)
        (df.write
           .format("delta")
           .mode("append")
           .option("mergeSchema", "true")
           .saveAsTable(TARGET_TABLE)
        )

        # Log success (prevents duplicates on rerun)
        log_ingest(source_file, file_size, file_mod_time, TARGET_TABLE, "SUCCESS")
        processed_keys.add(signature)

        # Optional: archive file after success (strongest dedupe guarantee)
        # if ARCHIVE_FOLDER:
        #     filename = source_file.split("/")[-1]
        #     archive_path = ARCHIVE_FOLDER.rstrip("/") + "/" + filename
        #     dbutils.fs.mv(source_file, archive_path)
        #     print(f"ARCHIVED: {archive_path}")

        # print(f"SUCCESS: {source_file} -> {TARGET_TABLE}")

    except Exception as e:
        print(f"FAILED: {source_file} -> {TARGET_TABLE}\n{e}")
        log_ingest(source_file, file_size, file_mod_time, TARGET_TABLE, "FAILED")
        raise

